# Assignment 2: training pipeline (hard hat compliance)

Runs README steps 3 to 6 end to end: download the dataset, describe it, finetune YOLO,
compare against two baselines, report per-class results, and sweep the decision
threshold against a cost matrix.

This notebook produces the **numbers and figures**. The interpretation and the reasoning
behind each figure belong in `REPORT.md`.

Run it with `uv run python main.py train`. Set `QUICK=1` in the environment for a
one-epoch smoke test. A quick run writes to `runs/quick/` and never touches
`weights/` or `outputs/`.

## 0. Setup

The first cell only does something on a Google Colab runtime (in the browser, or through
the Colab extension for VS Code). There the repo is not on the machine yet, so it clones
the fork, installs the two packages Colab lacks, and asks for the Roboflow key without
storing it. Locally and under `main.py` it does nothing.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/pilarguerreromorales/daiist-assignment-2.git"

if IN_COLAB:
    repo = "/content/daiist-assignment-2"
    if os.path.isdir(repo):
        subprocess.run(["git", "-C", repo, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, repo], check=True)
    os.chdir(repo)
    sys.path.insert(0, repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4,<8.5", "roboflow>=1.7"],
                   check=True)
    if not os.environ.get("ROBOFLOW_API_KEY"):
        from getpass import getpass
        os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API key: ")

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import matplotlib

IN_NOTEBOOK = "ipykernel" in sys.modules
if not IN_NOTEBOOK:
    matplotlib.use("Agg")  # run via main.py: save figures instead of opening windows

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from matplotlib.patches import Rectangle
from PIL import Image
from ultralytics import YOLO

from api_key import load_api_key

ROOT = Path.cwd()
pd.set_option("display.precision", 3)

In [ ]:
# Dataset: Roboflow Universe, used as published (not forked)
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "joseph-nelson", "hard-hat-workers", 5
DATA_DIR = ROOT / "datasets" / f"{RF_PROJECT}-v{RF_VERSION}"
CLASS_NAMES = ["head", "helmet"]
POSITIVE = "head"  # a worker without a helmet is the event we must catch
SPLITS = ["train", "valid", "test"]

# Training
QUICK = os.environ.get("QUICK") == "1"
BASE_MODEL = "yolo11n.pt"
EPOCHS = 1 if QUICK else 50
PATIENCE = 15
IMGSZ = 640
BATCH = 16
SEED = 0

# Evaluation
DET_CONF = 0.25  # confidence for box-level precision/recall and the detection confusion matrix
IOU_THRS = np.round(np.arange(0.5, 0.96, 0.05), 2)  # AP50 uses the first, mAP50-95 all ten
BOX = ["x1", "y1", "x2", "y2"]

if torch.cuda.is_available():
    DEVICE = 0
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

OUT_DIR = ROOT / "runs" / "quick" if QUICK else ROOT / "outputs"
WEIGHTS_DIR = ROOT / "runs" / "quick" if QUICK else ROOT / "weights"
PRETRAINED_DIR = ROOT / "runs" / "pretrained"  # COCO checkpoints, kept out of git
FIG_DIR = OUT_DIR / "figures"
for d in (OUT_DIR, WEIGHTS_DIR, PRETRAINED_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

COLORS = {"head": "tab:red", "helmet": "tab:green"}
print(f"device={DEVICE}  quick={QUICK}  outputs -> {OUT_DIR.relative_to(ROOT)}")

In [ ]:
def save_fig(fig, name):
    """Save every figure so the report and the app can reuse it."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


def save_json(obj, name):
    (OUT_DIR / name).write_text(json.dumps(obj, indent=2, default=float))

## 1. Download the dataset

Downloads `joseph-nelson/hard-hat-workers` version 5 (`raw_HeadHelmetClasses`): two
classes and no augmentation, keeping Roboflow's train/valid/test split. If the dataset is
already in `datasets/`, the download is skipped.

In [ ]:
if not (DATA_DIR / "data.yaml").exists():
    from roboflow import Roboflow

    rf = Roboflow(api_key=load_api_key())
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        "yolov11", location=str(DATA_DIR)
    )

rf_yaml = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
assert rf_yaml["names"] == CLASS_NAMES, rf_yaml["names"]

# Roboflow's data.yaml uses relative paths that Ultralytics resolves from the wrong folder,
# so write one with an absolute root (kept inside the gitignored datasets/ folder).
DATA_YAML = DATA_DIR / "data_local.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": dict(enumerate(CLASS_NAMES)),
}))
print(rf_yaml.get("roboflow"))

## 2. Describe the dataset (README step 3)

Instances per class, split sizes, image-level composition, and object size relative to the
image. These are computed from the label files, not taken from the Universe page.

In [ ]:
def read_split(split):
    """One row per box and one row per image for a split, from the YOLO label files."""
    boxes, images = [], []
    for img in sorted((DATA_DIR / split / "images").iterdir()):
        w, h = Image.open(img).size
        label = DATA_DIR / split / "labels" / f"{img.stem}.txt"
        lines = label.read_text().splitlines() if label.exists() else []
        counts = dict.fromkeys(CLASS_NAMES, 0)
        for line in filter(str.strip, lines):
            c, x, y, bw, bh = line.split()
            name = CLASS_NAMES[int(c)]
            counts[name] += 1
            x, y, bw, bh = map(float, (x, y, bw, bh))
            boxes.append(dict(
                split=split, image=img.name, cls=name, img_w=w, img_h=h,
                x1=(x - bw / 2) * w, y1=(y - bh / 2) * h, x2=(x + bw / 2) * w, y2=(y + bh / 2) * h,
                rel_area=bw * bh,
            ))
        images.append(dict(split=split, image=img.name, path=str(img), img_w=w, img_h=h,
                           **{f"n_{c}": n for c, n in counts.items()}))
    return boxes, images


parts = [read_split(s) for s in SPLITS]
gt = pd.DataFrame([b for bx, _ in parts for b in bx])
images = pd.DataFrame([i for _, im in parts for i in im])

# Side length of each box in pixels once the image is letterboxed to IMGSZ for training
scale = IMGSZ / np.maximum(gt.img_w, gt.img_h)
gt["side_px"] = np.sqrt((gt.x2 - gt.x1) * (gt.y2 - gt.y1)) * scale

images["has_head"] = images.n_head > 0
images["composition"] = np.select(
    [images.n_head > 0, images.n_helmet > 0], ["≥1 bare head", "all helmeted"], "no people"
)

In [ ]:
summary = pd.DataFrame({
    "images": images.groupby("split").size(),
    "head boxes": gt[gt.cls == "head"].groupby("split").size(),
    "helmet boxes": gt[gt.cls == "helmet"].groupby("split").size(),
    "images with ≥1 head": images.groupby("split").has_head.sum(),
    "share of images with ≥1 head": images.groupby("split").has_head.mean(),
    "median box area (head)": gt[gt.cls == "head"].groupby("split").rel_area.median(),
    "median box area (helmet)": gt[gt.cls == "helmet"].groupby("split").rel_area.median(),
}).loc[SPLITS]
summary.loc["total"] = summary.sum().where(~summary.columns.str.contains("share|median"))
summary.to_csv(OUT_DIR / "dataset_summary.csv")
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

inst = gt.groupby(["split", "cls"]).size().unstack().loc[SPLITS]
inst.plot.bar(ax=axes[0], color=[COLORS[c] for c in inst.columns], rot=0)
axes[0].set(title="Box instances per class and split", ylabel="boxes", xlabel="")
for cont in axes[0].containers:
    axes[0].bar_label(cont, fontsize=8)

comp = pd.crosstab(images.split, images.composition, normalize="index").loc[SPLITS]
comp.plot.barh(stacked=True, ax=axes[1], color={"≥1 bare head": "tab:red", "all helmeted": "tab:green",
                                                  "no people": "lightgrey"})
axes[1].set(title="Image-level composition per split", xlabel="share of images", ylabel="")
axes[1].legend(loc="lower right", fontsize=8)
save_fig(fig, "eda_class_balance")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bins = np.logspace(-5, 0, 40)
for c in reversed(CLASS_NAMES):  # helmet first so the smaller head histogram is drawn on top
    axes[0].hist(gt[gt.cls == c].rel_area, bins=bins, alpha=0.6, label=c, color=COLORS[c])
axes[0].set(xscale="log", title="Box area relative to image", xlabel="box area / image area", ylabel="boxes")
axes[0].axvline(0.01, color="k", ls=":", lw=1)
axes[0].legend()

bins = np.arange(0, 200, 4)
for c in reversed(CLASS_NAMES):
    axes[1].hist(gt[gt.cls == c].side_px.clip(upper=196), bins=bins, alpha=0.6, label=c, color=COLORS[c])
axes[1].set(title=f"Box side length after resize to {IMGSZ}px", xlabel="pixels", ylabel="boxes")
axes[1].axvline(32, color="k", ls=":", lw=1)
axes[1].legend()

heads = images[images.has_head].n_head
axes[2].hist(heads.clip(upper=20), bins=np.arange(1, 22) - 0.5, color=COLORS["head"])
axes[2].set(title="Bare heads per image (images with ≥1)", xlabel="heads (20 = 20 or more)", ylabel="images")
save_fig(fig, "eda_object_size")

size_bins = pd.cut(gt.side_px, [0, 16, 32, 64, np.inf], labels=["<16px", "16-32px", "32-64px", "≥64px"])
display(pd.crosstab([gt.split, gt.cls], size_bins).loc[SPLITS])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for s in SPLITS:
    d = images[images.split == s]
    ax.scatter(d.img_w, d.img_h, s=6, alpha=0.4, label=s)
ax.set(title="Image resolution", xlabel="width (px)", ylabel="height (px)")
ax.legend()
save_fig(fig, "eda_resolution")
display(images.groupby("split")[["img_w", "img_h"]].describe().loc[SPLITS].T)

In [ ]:
def draw(ax, image_row, truth=None, preds=None, title=None):
    """Image with ground truth (solid) and predictions (dashed, labelled with confidence)."""
    ax.imshow(Image.open(image_row.path))
    for frame, style in ((truth, "-"), (preds, "--")):
        if frame is None:
            continue
        for b in frame[frame.image == image_row.image].itertuples():
            ax.add_patch(Rectangle((b.x1, b.y1), b.x2 - b.x1, b.y2 - b.y1, fill=False,
                                   ec=COLORS[b.cls], lw=1.5, ls=style))
            if style == "--":
                ax.text(b.x1, b.y1, f"{b.conf:.2f}", color="white", fontsize=7,
                        bbox=dict(fc=COLORS[b.cls], ec="none", pad=1))
    ax.set_title(title or image_row.image[:30], fontsize=8)
    ax.axis("off")


def grid(rows, title, name, truth=None, preds=None, titles=None, ncols=4):
    rows = list(rows.itertuples())
    if not rows:
        print(f"{title}: nothing to show")
        return
    nrows = int(np.ceil(len(rows) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.4 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, (ax, r) in enumerate(zip(axes.flat, rows)):
        draw(ax, r, truth, preds, titles[i] if titles else None)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, name)


test_imgs = images[images.split == "test"].reset_index(drop=True)
test_gt = gt[gt.split == "test"].reset_index(drop=True)
grid(pd.concat([test_imgs[test_imgs.has_head].sample(4, random_state=SEED),
                test_imgs[~test_imgs.has_head].sample(4, random_state=SEED)]),
     "Test samples, ground truth (top: ≥1 bare head, bottom: all helmeted)", "eda_samples", truth=test_gt)

## 3. Models and baselines (README step 4)

All three are scored on the same **test split** with the same matching code: a prediction
is correct if it has the right class and IoU ≥ 0.5 with an unused ground-truth box.

| Model | What it is |
|---|---|
| **Finetuned** | `yolo11n` from COCO weights, trained on the train split, best epoch picked on valid |
| **COCO zero-shot** | The same pretrained `yolo11n` with no training. COCO has no head or helmet class, so its `person` boxes are read as `head` ("a person it cannot vouch for"). All other COCO classes are dropped. |
| **Predict nothing** | Never flags anything, so every image passes |

In [ ]:
def get_pretrained(name):
    """COCO checkpoint stored under runs/ so it never lands in weights/ or the repo root."""
    path = PRETRAINED_DIR / name
    if not path.exists():
        src = Path(YOLO(name).ckpt_path)  # Ultralytics downloads into the working directory
        shutil.copy(src, path)
        if src.resolve().parent == ROOT:
            src.unlink()
    return path


def predict_split(model, split="test", conf=0.001):
    """Every box the model outputs on a split, down to a very low confidence, so thresholds can be swept later."""
    rows = []
    for r in model.predict(str(DATA_DIR / split / "images"), conf=conf, imgsz=IMGSZ, batch=BATCH,
                           device=DEVICE, stream=True, verbose=False):
        name = Path(r.path).name
        for xyxy, c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.int().tolist(), r.boxes.conf.tolist()):
            rows.append(dict(image=name, cls=r.names[c], conf=s, **dict(zip(BOX, xyxy))))
    return pd.DataFrame(rows, columns=["image", "cls", "conf", *BOX])

### Finetuning

The settings are kept in one place (the config cell) so the report can quote them.
`yolo11n` is the smaller of the two allowed models. Training images are letterboxed to
640px, which is the size the head-size histogram above is measured at.

In [ ]:
train_cfg = dict(
    data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, seed=SEED,
    device=DEVICE, project=str(ROOT / "runs"), name="quick_train" if QUICK else "hardhat", exist_ok=True,
    fraction=0.05 if QUICK else 1.0, plots=True, verbose=False,
)
model = YOLO(get_pretrained(BASE_MODEL))
model.train(**train_cfg)

best_src = Path(model.trainer.best)
WEIGHTS = WEIGHTS_DIR / f"hardhat_{Path(BASE_MODEL).stem}.pt"
shutil.copy(best_src, WEIGHTS)
save_json({**train_cfg, "base_model": BASE_MODEL, "device": str(DEVICE),
           "torch": torch.__version__, "best_epoch_weights": str(WEIGHTS.relative_to(ROOT))},
          "training_config.json")
print("saved", WEIGHTS.relative_to(ROOT))

In [ ]:
results_png = best_src.parent.parent / "results.png"
if results_png.exists():
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(results_png))
    ax.axis("off")
    ax.set_title("Training curves (Ultralytics)")
    save_fig(fig, "training_curves")

In [ ]:
finetuned = YOLO(WEIGHTS)
coco = YOLO(get_pretrained(BASE_MODEL))

raw_coco = predict_split(coco)
preds = {
    "finetuned": predict_split(finetuned),
    "coco_zero_shot": raw_coco[raw_coco.cls == "person"].assign(cls="head"),
    "predict_nothing": pd.DataFrame(columns=["image", "cls", "conf", *BOX]).astype({"conf": float}),
}
{k: len(v) for k, v in preds.items()}

## 4. Per-class evaluation (README step 5)

In [ ]:
def iou_matrix(a, b):
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area = lambda x: (x[:, 2:] - x[:, :2]).prod(1)
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def match_class(p, truth, cls, iou_thr=0.5):
    """Greedy, highest confidence first: each prediction of `cls` takes the best unused
    ground-truth box of `cls` with IoU >= iou_thr. Returns (predictions with a tp flag,
    ground truth with a hit flag)."""
    p = p[p.cls == cls].reset_index(drop=True)
    g = truth[truth.cls == cls].reset_index(drop=True)
    tp, hit = np.zeros(len(p), bool), np.zeros(len(g), bool)
    g_idx = g.groupby("image").groups
    for img, d in p.groupby("image"):
        if img not in g_idx:
            continue
        gi = np.asarray(g_idx[img])
        d = d.sort_values("conf", ascending=False)
        ious = iou_matrix(d[BOX].to_numpy(), g.loc[gi, BOX].to_numpy())
        for row, iou in zip(d.index, ious):
            iou = np.where(hit[gi], 0.0, iou)
            j = iou.argmax()
            if iou[j] >= iou_thr:
                hit[gi[j]] = tp[row] = True
    return p.assign(tp=tp), g.assign(hit=hit)


def average_precision(tp, conf, n_gt):
    """Area under the interpolated precision-recall curve (101-point, as in COCO)."""
    if n_gt == 0:
        return np.nan
    if len(tp) == 0:
        return 0.0
    tp = tp[np.argsort(-conf)]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    recall = np.concatenate([[0], ctp / n_gt, [1]])
    precision = np.concatenate([[1], ctp / (ctp + cfp), [0]])
    precision = np.flip(np.maximum.accumulate(np.flip(precision)))
    x = np.linspace(0, 1, 101)
    return np.trapezoid(np.interp(x, recall, precision), x)


def per_class(p, truth, conf=DET_CONF):
    rows = []
    for c in CLASS_NAMES:
        aps = []
        for t in IOU_THRS:
            m, g = match_class(p, truth, c, t)
            aps.append(average_precision(m.tp.to_numpy(), m.conf.to_numpy(), len(g)))
            if t == 0.5:
                kept = m[m.conf >= conf]
                tp50 = kept.tp.sum()
                rows.append(dict(cls=c, instances=len(g), predictions=len(kept),
                                 precision=tp50 / len(kept) if len(kept) else np.nan,
                                 recall=tp50 / len(g)))
        rows[-1].update(AP50=aps[0], **{"AP50-95": np.mean(aps)})
    return pd.DataFrame(rows).set_index("cls")


results = {name: per_class(p, test_gt) for name, p in preds.items()}
table = pd.concat(results, names=["model"])
overall = table.groupby("model")[["AP50", "AP50-95"]].mean().rename(columns=lambda c: "m" + c)
table.to_csv(OUT_DIR / "per_class_metrics.csv")
print(f"Box-level precision/recall at confidence ≥ {DET_CONF}; AP over all confidences.")
display(table)
display(overall)

In [ ]:
# Cross-check against Ultralytics' own validator on the same test split (finetuned model only).
# Its precision/recall are taken at the confidence that maximises F1, so they differ from the table above.
val = finetuned.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                    project=str(ROOT / "runs"), name="test_eval", exist_ok=True, plots=True, verbose=False)
ul = pd.DataFrame({
    "precision": val.box.p, "recall": val.box.r, "AP50": val.box.ap50, "AP50-95": val.box.ap,
}, index=[CLASS_NAMES[i] for i in val.box.ap_class_index])
ul.loc["all"] = [val.box.mp, val.box.mr, val.box.map50, val.box.map]
display(ul)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
table["AP50"].unstack("cls").loc[list(preds)].plot.bar(ax=ax, rot=0, color=[COLORS[c] for c in CLASS_NAMES])
ax.set(title="AP50 per class on the test split", ylabel="AP50", ylim=(0, 1), xlabel="")
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=8)
save_fig(fig, "ap50_per_class")

In [ ]:
def detection_confusion(p, truth, conf=DET_CONF, iou_thr=0.5):
    """Rows = truth, columns = prediction. 'background' row = predicted box with no
    matching object; 'background' column = object the model missed."""
    labels = CLASS_NAMES + ["background"]
    cm = pd.DataFrame(0, index=labels, columns=labels)
    p = p[p.conf >= conf]
    p_by, g_by = dict(list(p.groupby("image"))), dict(list(truth.groupby("image")))
    for img in set(p_by) | set(g_by):
        d = p_by.get(img, p.iloc[:0]).sort_values("conf", ascending=False)
        g = g_by.get(img, truth.iloc[:0])
        used = np.zeros(len(g), bool)
        ious = iou_matrix(d[BOX].to_numpy(), g[BOX].to_numpy()) if len(d) and len(g) else np.zeros((len(d), 0))
        for (_, pr), iou in zip(d.iterrows(), ious):
            iou = np.where(used, 0.0, iou)
            if len(iou) and iou.max() >= iou_thr:
                j = iou.argmax()
                used[j] = True
                cm.loc[g.cls.iloc[j], pr.cls] += 1
            else:
                cm.loc["background", pr.cls] += 1
        for c in g.cls[~used]:
            cm.loc[c, "background"] += 1
    cm.loc["background", "background"] = 0  # undefined for detection
    return cm


def plot_cm(cm, title, ax, normalise_rows=True):
    shown = cm.div(cm.sum(axis=1).replace(0, 1), axis=0) if normalise_rows else cm
    ax.imshow(shown, cmap="Blues", vmin=0, vmax=1 if normalise_rows else None)
    ax.set_xticks(range(len(cm.columns)), cm.columns)
    ax.set_yticks(range(len(cm.index)), cm.index)
    ax.set(xlabel="predicted", ylabel="true", title=title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            txt = f"{cm.iat[i, j]}" + (f"\n{shown.iat[i, j]:.0%}" if normalise_rows else "")
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if normalise_rows and shown.iat[i, j] > 0.6 else "black")


det_cms = {name: detection_confusion(p, test_gt) for name, p in preds.items()}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_cm(det_cms["finetuned"], f"Finetuned: box confusion matrix (conf ≥ {DET_CONF})", axes[0])
plot_cm(det_cms["coco_zero_shot"], "COCO zero-shot (person → head)", axes[1])
fig.tight_layout()
save_fig(fig, "confusion_matrix_detection")

### Where it fails

Head recall by object size, the worst missed heads, and the most confident false `head`
boxes.

In [ ]:
_, head_hits = match_class(preds["finetuned"][preds["finetuned"].conf >= DET_CONF], test_gt, "head")
head_hits["size"] = pd.cut(test_gt[test_gt.cls == "head"].side_px.to_numpy(), [0, 16, 32, 64, np.inf],
                           labels=["<16px", "16-32px", "32-64px", "≥64px"])
by_size = head_hits.groupby("size", observed=False).hit.agg(recall="mean", heads="size")

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(by_size.index.astype(str), by_size.recall, color=COLORS["head"])
ax.bar_label(bars, labels=[f"{r:.0%}\n(n={n})" for r, n in zip(by_size.recall, by_size.heads)], fontsize=8)
ax.set(title=f"Finetuned head recall by size (conf ≥ {DET_CONF})", ylabel="recall", ylim=(0, 1.15),
       xlabel=f"head side length at {IMGSZ}px")
save_fig(fig, "failure_recall_by_size")
display(by_size)

In [ ]:
ft = preds["finetuned"][preds["finetuned"].conf >= DET_CONF]
missed = head_hits[~head_hits.hit].groupby("image").size().sort_values(ascending=False)
grid(test_imgs.set_index("image").loc[missed.index[:8]].reset_index(),
     "Most missed bare heads (solid = truth, dashed = prediction)", "failure_missed_heads",
     truth=test_gt, preds=ft, titles=[f"{n} missed heads" for n in missed.values[:8]])

ft_matched, _ = match_class(ft, test_gt, "head")
false_heads = ft_matched[~ft_matched.tp].sort_values("conf", ascending=False).drop_duplicates("image")
grid(test_imgs.set_index("image").loc[false_heads.image[:8]].reset_index(),
     "Most confident false 'head' boxes", "failure_false_heads", truth=test_gt, preds=ft,
     titles=[f"false head, conf {c:.2f}" for c in false_heads.conf[:8]])

## 5. From boxes to a decision per image (README step 6)

One **case** is one image (one camera frame). Its score is the highest `head` confidence in
it. The case is flagged when that score is ≥ the threshold. Ground truth: the image
contains at least one bare head.

The test split has about 20% positive images. Costs are computed with the rates measured
on the test split (TPR, FPR), re-weighted to the base rate assumed below, so the result
does not depend on the test split's class mix.

In [ ]:
y_true = test_imgs.has_head.to_numpy()


def image_scores(p):
    best = p[p.cls == POSITIVE].groupby("image").conf.max()
    return test_imgs.image.map(best).fillna(0.0).to_numpy()


scores = {name: image_scores(p) for name, p in preds.items()}
pd.DataFrame({"image": test_imgs.image, "has_head": y_true, "n_head": test_imgs.n_head,
              **{f"score_{k}": v for k, v in scores.items()}}).to_csv(OUT_DIR / "test_image_scores.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, 1, 41)
axes[0].hist(scores["finetuned"][~y_true], bins=bins, alpha=0.6, label="all helmeted", color="tab:green")
axes[0].hist(scores["finetuned"][y_true], bins=bins, alpha=0.6, label="≥1 bare head", color="tab:red")
axes[0].set(title="Finetuned: image score distribution", xlabel="max head confidence", ylabel="images")
axes[0].legend()

for name, s in scores.items():
    thr = np.unique(np.concatenate([[0, 1.01], s]))
    tpr = [(s[y_true] >= t).mean() for t in thr]
    fpr = [(s[~y_true] >= t).mean() for t in thr]
    axes[1].plot(fpr, tpr, label=name)
axes[1].plot([0, 1], [0, 1], "k:", lw=1)
axes[1].set(title="Image-level ROC", xlabel="false positive rate (compliant images flagged)",
            ylabel="true positive rate (violations caught)")
axes[1].legend()
save_fig(fig, "image_level_scores")

### Cost matrix and base rate

One case is one camera snapshot, checked once. Costs are in euros per case; the reasoning
behind each figure is in REPORT.md.

In [ ]:
# Safety technician: ~€30k gross/yr + ~32% employer social security ≈ €40k / ~1,750 h ≈ €23/h
CHECK_MINUTES = 5
HOURLY_COST = 23.0
CHECK_COST = round(CHECK_MINUTES / 60 * HOURLY_COST)  # ≈ €2

COST = {
    "FP": CHECK_COST,  # compliant snapshot flagged: one wasted check
    "TP": CHECK_COST,  # violation caught: the same check
    "TN": 0.0,         # compliant snapshot passed: nothing happens
    "FN": 200.0,       # violation missed: 100x a false alarm (≈ 1-in-1,000 chance of a €200k head injury)
}
BASE_RATE = 0.05  # assumed share of real-site snapshots with ≥1 bare head (dataset has ~20%)
THRESHOLDS = np.round(np.arange(0, 1.0001, 0.01), 2)


def rates(s, thr):
    return (s[y_true] >= thr).mean(), (s[~y_true] >= thr).mean()


def cost_per_1000(tpr, fpr, base_rate=BASE_RATE, cost=COST):
    pos = base_rate * (tpr * cost["TP"] + (1 - tpr) * cost["FN"])
    neg = (1 - base_rate) * (fpr * cost["FP"] + (1 - fpr) * cost["TN"])
    return 1000 * (pos + neg)


rows = []
for name, s in scores.items():
    for t in THRESHOLDS:
        tpr, fpr = rates(s, t)
        rows.append(dict(model=name, threshold=t, tpr=tpr, fpr=fpr,
                         flagged_share=BASE_RATE * tpr + (1 - BASE_RATE) * fpr,
                         cost_per_1000=cost_per_1000(tpr, fpr)))
sweep = pd.DataFrame(rows)
sweep.to_csv(OUT_DIR / "cost_sweep.csv", index=False)

ft_sweep = sweep[sweep.model == "finetuned"].set_index("threshold")
best_thr = ft_sweep.cost_per_1000.idxmin()
references = {
    "pass everything (no model)": cost_per_1000(0, 0),
    "flag everything (person checks all)": cost_per_1000(1, 1),
}
display(pd.Series({**references,
                   **{f"{m} at its best threshold": sweep[sweep.model == m].cost_per_1000.min() for m in scores}},
                  name="cost per 1,000 cases (€)"))
print(f"Finetuned minimum-cost threshold: {best_thr:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for name in ["finetuned", "coco_zero_shot"]:
    d = sweep[sweep.model == name]
    ax.plot(d.threshold, d.cost_per_1000, label=name, lw=2 if name == "finetuned" else 1)
for (label, v), ls in zip(references.items(), ["--", ":"]):
    ax.axhline(v, color="grey", ls=ls, lw=1, label=label)
ax.scatter([best_thr], [ft_sweep.cost_per_1000[best_thr]], color="k", zorder=5,
           label=f"chosen threshold {best_thr:.2f} (€{ft_sweep.cost_per_1000[best_thr]:,.0f})")
ax.set(title=f"Expected cost per 1,000 cases vs threshold (base rate {BASE_RATE:.0%})",
       xlabel="confidence threshold for 'bare head'", ylabel="€ per 1,000 cases")
ax.legend(fontsize=8)
save_fig(fig, "cost_vs_threshold")

In [ ]:
def image_confusion(s, thr):
    pred = s >= thr
    return pd.DataFrame([[(y_true & pred).sum(), (y_true & ~pred).sum()],
                         [(~y_true & pred).sum(), (~y_true & ~pred).sum()]],
                        index=["true: ≥1 head", "true: all helmeted"],
                        columns=["flagged", "passed"])


cm_img = image_confusion(scores["finetuned"], best_thr)
tpr, fpr = rates(scores["finetuned"], best_thr)
cm_1000 = pd.DataFrame([[1000 * BASE_RATE * tpr, 1000 * BASE_RATE * (1 - tpr)],
                        [1000 * (1 - BASE_RATE) * fpr, 1000 * (1 - BASE_RATE) * (1 - fpr)]],
                       index=cm_img.index, columns=cm_img.columns).round(1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
plot_cm(cm_img, f"Test split, threshold {best_thr:.2f}", axes[0])
plot_cm(cm_1000, f"Per 1,000 cases at base rate {BASE_RATE:.0%}", axes[1], normalise_rows=False)
fig.tight_layout()
save_fig(fig, "confusion_matrix_image_level")

### Sensitivity of the chosen threshold

How the minimum-cost threshold moves when the FN:FP cost ratio and the base rate change.

In [ ]:
def best_threshold(base_rate, cost):
    c = [cost_per_1000(*rates(scores["finetuned"], t), base_rate, cost) for t in THRESHOLDS]
    return THRESHOLDS[int(np.argmin(c))], min(c)


ratios = np.unique(np.round(np.logspace(0, 3, 25)))
base_rates = [0.01, 0.02, 0.05, 0.1, 0.2, 0.4]
sens = pd.DataFrame([dict(ratio=r, base_rate=b, threshold=best_threshold(b, {**COST, "FN": COST["FP"] * r})[0])
                     for r in ratios for b in base_rates])
sens.to_csv(OUT_DIR / "threshold_sensitivity.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
for b, d in sens.groupby("base_rate"):
    ax.plot(d.ratio, d.threshold, marker=".", label=f"base rate {b:.0%}")
ax.axvline(COST["FN"] / COST["FP"], color="k", ls=":", lw=1, label="current FN:FP ratio")
ax.set(xscale="log", title="Minimum-cost threshold vs cost ratio and base rate",
       xlabel="cost of a missed violation / cost of a false alarm", ylabel="best threshold")
ax.legend(fontsize=8)
save_fig(fig, "threshold_sensitivity")

## 6. Save what the app needs

In [ ]:
save_json({
    "dataset": {"workspace": RF_WORKSPACE, "project": RF_PROJECT, "version": RF_VERSION},
    "weights": str(WEIGHTS.relative_to(ROOT)),
    "positive_class": POSITIVE,
    "class_names": CLASS_NAMES,
    "imgsz": IMGSZ,
    "det_conf": DET_CONF,
    "cost": COST,
    "base_rate": BASE_RATE,
    "threshold": float(best_thr),
    "cost_per_1000_at_threshold": float(ft_sweep.cost_per_1000[best_thr]),
    "references_per_1000": references,
    "overall": overall.to_dict(orient="index"),
    "detection_confusion": {k: v.to_dict() for k, v in det_cms.items()},
}, "operating_point.json")
print(sorted(p.name for p in OUT_DIR.iterdir()))

### Colab only: bundle the results

Zips the trained weights and `outputs/` so they can be copied back into the local repo and
committed. Does nothing locally.

In [ ]:
if IN_COLAB:
    bundle = Path("/content/hardhat_results.zip")
    subprocess.run(["zip", "-qr", str(bundle), WEIGHTS.relative_to(ROOT), OUT_DIR.relative_to(ROOT)],
                   cwd=ROOT, check=True)
    print(f"bundle: {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as e:  # the VS Code extension may not support browser downloads
        print(f"automatic download unavailable ({e}); fetch {bundle} from the Colab file browser")